In [ ]:
code_dict = {
"007301":"半导体",
"012832":"新能源",
"007883":"医药",
"012700":"证券",
"012043":"酒",
"009180":"消费",
"012323":"医疗",

"007339":"沪深300",
"014345":"中证500",
"001593":"创业",

"009052":"中证红利",

"008702":"黄金"
}

In [ ]:
import akshare as ak
import pandas as pd
from datetime import datetime
import os

# ===== 日期文件名 =====
today = datetime.today().strftime("%Y-%m-%d")

os.makedirs("data", exist_ok=True)

csv_path = f"data/fund_nav_{today}.csv"

# ===== 如果已存在则跳过 =====
if os.path.exists(csv_path):
    print("Today's data already exists, skip download:")
    print(csv_path)
else:
    
    print("Start downloading fund data...")
    
    data_list = []
    
    for code, name in code_dict.items():
    
        print("downloading:", name)
    
        df = ak.fund_open_fund_info_em(
            symbol=code,
            indicator="单位净值走势"
        )
    
        df = df.rename(columns={
            "净值日期": "date",
            "单位净值": "nav"
        })
    
        df["fund"] = name
        df["code"] = code
    
        data_list.append(df[["date", "code", "fund", "nav"]])
    
    
    # ===== 合并数据 =====
    data = pd.concat(data_list)
    
    data["date"] = pd.to_datetime(data["date"]).dt.date
    
    data = data.sort_values(["date", "code"]).reset_index(drop=True)
    
    print(data.head())
    
    
    # ===== 保存 =====
    data.to_csv(csv_path, index=False)
    
    print("Saved to:", csv_path)

In [ ]:
data["return"] = data.groupby("code")["nav"].pct_change()

In [ ]:
print(type(data))
print(data.head())
print(data.dtypes)
print(data[data["code"] == "007301"].head())